# NB 1.2 &mdash; El teu primer model, de principi a fi

**MP 5134** &mdash; UT1 &mdash; *Versió California Housing*

---
### Com funciona aquest notebook

Al revés del que esperes.

Primer executaràs un bloc de codi que entrena un model complet i et donarà un
número. **No entendràs res del que passa, i està bé així.** Després el
desmuntarem peça a peça.

Es fa així a propòsit. Si comencéssim per la teoria, arribaries al codi tres
sessions més tard sense saber per a què servia. Aquesta manera és més incòmoda
els primers deu minuts i molt més eficient la resta del curs.

## 1. El bloc complet

Executa la cel·la següent sense intentar entendre-la.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

URL_DADES = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
              "master/datasets/housing/housing.csv")

df = pd.read_csv(URL_DADES)

FEATURES = ["longitude", "latitude", "housing_median_age", "total_rooms",
            "total_bedrooms", "population", "households", "median_income"]
data = df[FEATURES + ["median_house_value"]].dropna()

X = data[FEATURES]
y = data["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Puntuació del model: {model.score(X_test, y_test):.3f}")

Ja està. Acabes d'entrenar un model d'aprenentatge automàtic i de mesurar com de
bé funciona sobre dades que no havia vist mai.

Són quinze línies. La resta del notebook consisteix a entendre-les.

## 2. Pas a pas

### 2.1 Separar les entrades de la sortida

Tot problema supervisat es divideix sempre en dues peces:

- **X**: la taula de característiques, el que li donem al model.
- **y**: el vector objectiu, el que li demanem que endevini.

La convenció de posar `X` en majúscula i `y` en minúscula ve de les matemàtiques
i la trobaràs a tota la documentació i a tots els llibres: `X` és una taula de
dues dimensions i `y` és un vector d'una.

In [ ]:
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print()
print("Columnes de X:", list(X.columns))

Fixa't en tres decisions que hem pres en aquestes línies, i que no són innocents.

**`median_house_value` no és a `X`.** Seria l'error més greu possible: donar-li al
model la resposta com a entrada. En diem **fuga d'informació** (*data leakage*) i
és una de les causes més freqüents de resultats espectaculars que després no
funcionen. Hi tornarem a la UT8 i a la UT10.

**`ocean_proximity` tampoc hi és.** No perquè no serveixi, que serviria molt, sinó
perquè és text i els algorismes només accepten números. Convertir-la és feina de
la UT3.

**Hem fet `dropna()`.** Els 207 buits de `total_bedrooms` que vam veure al NB 1.1
no desapareixen sols: sense aquesta crida, la cel·la anterior hauria fallat.

### 2.2 Separar entrenament i prova

Aquesta és la idea més important de tot el mòdul, i la més fàcil de saltar-se.

Si entrenem el model amb totes les dades i després li preguntem sobre aquestes
mateixes dades, el resultat no ens diu res. És com corregir un examen a algú que
ha vist les respostes: no mesures si ha après, mesures si té memòria.

Per això apartem un tros de les dades **abans** d'entrenar, i no el tocarem fins
al moment d'avaluar.

In [ ]:
print(f"Mostres per entrenar: {len(X_train)}")
print(f"Mostres per provar:   {len(X_test)}  ({100 * len(X_test) / len(X):.0f} %)")

Dos paràmetres de `train_test_split` que veuràs cada dia:

`test_size=0.2` aparta el 20% de les dades per provar. El valor habitual està
entre el 10% i el 30%, i la **UT10** explicarà per què.

`random_state=42` fixa l'atzar. Sense això, cada execució faria una partició
diferent i obtindries un número lleugerament diferent cada vegada, cosa que fa
impossible comparar. El 42 no té cap significat tècnic; és una broma heretada
que s'ha convertit en costum.

### 2.3 L'estimador és un objecte

Aquí és on el que ja sabeu de programació orientada a objectes us dóna avantatge.

A scikit-learn, **un model és una classe**. Quan escrius `LinearRegression()`
estàs instanciant un objecte, igual que faries amb qualsevol altra classe.

In [ ]:
new_model = LinearRegression()

print("Tipus:", type(new_model))
print()
print("Paràmetres de configuració:")
for key, value in new_model.get_params().items():
    print(f"  {key} = {value}")

Aquest objecte acabat de crear **encara no sap res**. És un model buit, amb la
seva configuració però sense cap coneixement de les nostres dades.

El que li dóna contingut és el mètode `fit()`. I aquí hi ha la clau conceptual:
`fit()` **modifica l'estat intern de l'objecte**. Abans de cridar-lo, l'objecte
no té apresos els seus paràmetres; després, sí.

In [ ]:
print("Té coeficients abans de fit()?", hasattr(new_model, "coef_"))

new_model.fit(X_train, y_train)

print("Té coeficients després de fit()?", hasattr(new_model, "coef_"))
print()
for name, coef in zip(FEATURES, new_model.coef_):
    print(f"  {name:>20} : {coef: 12.2f}")

Aquests números són el que el model ha après. Cadascun diu quant puja o baixa la
predicció quan aquella variable augmenta en una unitat.

Mira el de `median_income`: és enorme comparat amb els altres. I el de
`total_rooms` és minúscul. Això **no** vol dir que els ingressos importin més:
vol dir que estan mesurats en unitats molt diferents. Els ingressos van de 0 a
15, i les habitacions arriben a desenes de milers.

Aquest problema de les escales té solució i es diu **escalat**. El veurem a la
UT3, i a la UT6 comprovarem que hi ha algorismes que directament no funcionen
sense ell.

Una convenció de scikit-learn que val la pena conèixer: **els atributs que
acaben amb guió baix (`coef_`, `intercept_`) són els que s'han après durant
l'entrenament**. Els que no en porten són configuració que has posat tu.

Aquesta interfície és idèntica per a tots els models de la biblioteca. Canviaràs
`LinearRegression` per `RandomForestRegressor` o per `SVC` i les tres crides
seran exactament les mateixes. Això és el que fa que puguem comparar algorismes
amb tanta facilitat durant tot el curs.

### 2.4 Predir i puntuar

Tres mètodes i ja tens el cicle complet:

| Mètode | Què fa |
|---|---|
| `fit(X, y)` | aprèn a partir dels exemples |
| `predict(X)` | dóna prediccions per a mostres noves |
| `score(X, y)` | mesura com de bé ho ha fet |

In [ ]:
predictions = model.predict(X_test)

comparison = pd.DataFrame({
    "real": y_test.values[:10].round(0),
    "predit": predictions[:10].round(0),
})
comparison["error"] = (comparison["predit"] - comparison["real"]).round(0)
comparison

Aquesta taula és més informativa que qualsevol mètrica. Mira els errors.

És important que t'hi acostumis des d'ara: **abans de mirar el número global,
mira uns quants casos concrets**. Els números globals amaguen problemes que els
exemples individuals ensenyen de seguida.

N'hi haurà de petits i algun d'escandalós. Si hi trobes alguna predicció
negativa, pensa un moment què significa: un preu negatiu no existeix, però al
model ningú no li ho ha dit.

In [ ]:
print(f"R2 sobre el conjunt de prova: {model.score(X_test, y_test):.3f}")

El número que dóna `score()` en un problema de regressió és el **R quadrat**.
Interpretació ràpida:

- **1.0** seria una predicció perfecta.
- **0.0** vol dir que el model no ho fa millor que dir sempre la mitjana.
- **negatiu** vol dir que ho fa pitjor que dir sempre la mitjana.

No et quedis amb la fórmula, que la veurem a la UT2. Queda't amb la idea: el R2
compara el teu model amb el model més ximple possible.

## 3. El model de referència

I aquí ve la pregunta que separa qui entén el que fa de qui només executa
cel·les: **aquest número és bo?**

No es pot respondre sense un punt de comparació. Construïm-lo.

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

print(f"Referència (diu sempre la mitjana): {baseline.score(X_test, y_test):.3f}")
print(f"Regressió lineal:                   {model.score(X_test, y_test):.3f}")

`DummyRegressor` no aprèn res: diu sempre el mateix número. Serveix per saber
quant val la pena l'esforç.

Aquí la millora és clara i el model aporta valor de veritat. Però l'hàbit
d'establir sempre una referència abans de celebrar un resultat t'estalviarà
disgustos, i a la secció següent en veuràs un cas.

## 4. El mateix esquema, ara amb classificació

Canviem de problema: en lloc del preu exacte, volem saber només **si el districte
és car**.

Observa que les línies clau són idèntiques. Només canvia la variable objectiu i
la classe del model.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

THRESHOLD = 350_000
data_c = df[FEATURES].dropna()
yc = (df.loc[data_c.index, "median_house_value"] >= THRESHOLD).astype(int)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    data_c, yc, test_size=0.2, random_state=42
)

classifier = DecisionTreeClassifier(max_depth=4, random_state=42)
classifier.fit(Xc_train, yc_train)

print(f"Encerts del classificador: {classifier.score(Xc_test, yc_test):.3f}")

Un percentatge d'encerts alt. Sembla un èxit.

Abans de creure-t'ho, aplica el que acabes d'aprendre: construeix la
referència.

In [ ]:
from sklearn.dummy import DummyClassifier

ref_c = DummyClassifier(strategy="most_frequent")
ref_c.fit(Xc_train, yc_train)

print(f"Model que diu sempre 'no és car': {ref_c.score(Xc_test, yc_test):.3f}")
print(f"El nostre arbre de decisió:       {classifier.score(Xc_test, yc_test):.3f}")

### La trampa

Un model que no mira absolutament cap dada, que diu sempre el mateix, encerta
gairebé tant com el nostre arbre. La diferència és de pocs punts.

Això no vol dir que l'arbre sigui inútil. Vol dir que **el percentatge d'encerts
és una mètrica inadequada per a aquest problema**, perquè els districtes cars són
una minoria.

Pensa-ho en termes pràctics: si aquest model l'hagués de fer servir una
immobiliària per detectar zones cares, un model que no en detecta cap no li
serveix de res per molt que encerti el 86% de les vegades.

Aquesta és exactament la situació que treballarem a fons a la **UT4**, quan veurem
la matriu de confusió i mètriques com el *recall*. De moment queda't amb la
desconfiança: **un número alt no vol dir un bon model**.

## 5. Exercici de lectura de codi

Torna al bloc de la secció 1 i respon per escrit, sense executar res:

**1.** Quina línia fa que el model aprengui?

**2.** Quina línia garanteix que l'avaluació sigui honesta?

**3.** Què passaria si `FEATURES` inclogués `"median_house_value"`? Quin valor
donaria `score()` i per què això seria un error greu?

**4.** Si canvies `random_state=42` per `random_state=7`, canviarà el resultat?
Molt o poc? Prova-ho després d'haver respost.

**5.** Reescriu el bloc canviant `LinearRegression` per `DecisionTreeRegressor`
(l'has d'importar de `sklearn.tree`). Quantes línies has hagut de tocar? Què et
diu això sobre el disseny de scikit-learn?

**6.** Torna al NB 1.1 i recorda el truncament dels 500.001 dòlars. Com creus que
afecta el R2 que acabes de calcular? Perjudica el model o l'afavoreix?

## 6. Per al debat de classe

Pensa dos problemes del teu entorn (feina, aficions, el que sigui) que es
podrien plantejar com a aprenentatge automàtic.

Per a cadascun, defineix:

- Què seria una **mostra**?
- Quines serien les **característiques**?
- Quina seria la **variable objectiu**?
- És un problema de **regressió** o de **classificació**?

I una última, la més difícil: **d'on sortirien les dades?** En la pràctica
professional aquesta acostuma a ser la pregunta que decideix si un projecte és
viable o no.